# PAI102: Data Pipeline Using CSV, JSON, Avro, and Parquet


### Objective
Build a simple data pipeline to understand data types and common file formats:
- CSV
- JSON
- Avro
- Parquet



### Libraries Required
pandas pyarrow fastavro



In [1]:
%pip install pandas pyarrow fastavro

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.4 MB/s eta 0:00:00


## Task 1: Create Sample Dataset

In [2]:
# Import the pandas library, which is used for data manipulation and analysis
# It provides data structures like DataFrame for handling tabular data
import pandas as pd
# Create a list("data") of dictionaries
# Each dictionary represents one student record (i.e., one row in the table)
# Keys of the dictionary correspond to column names
data = [
    {"student_id": 1, "name": "Amit", "department": "CSE", "marks": 85.5, "passed": True},
    {"student_id": 2, "name": "Riya", "department": "ECE", "marks": 72.0, "passed": True},
    {"student_id": 3, "name": "Karan", "department": "ME", "marks": 48.0, "passed": False},
    {"student_id": 4, "name": "Sneha", "department": "CSE", "marks": 91.2, "passed": True},
]
# Convert the list of dictionaries into a pandas DataFrame
# The DataFrame organizes the data into rows and columns for easy analysis
df = pd.DataFrame(data)
# Display the DataFrame
# This shows the structured tabular representation of the student data
df


,student_id,name,department,marks,passed
0,1,Amit,CSE,85.5,True
1,2,Riya,ECE,72.0,True
2,3,Karan,ME,48.0,False
3,4,Sneha,CSE,91.2,True


## Task 2: Save Data as CSV

In [4]:

# Export the DataFrame 'df' to a CSV (Comma-Separated Values) file
# The file will be saved in the current working directory with the name "students.csv"
# index=False ensures that the DataFrame index is not written as a separate column in the CSV file
df.to_csv("students.csv", index=False)

# Print a confirmation message to indicate that the CSV file has been successfully created
print("CSV file created")



CSV file created


## Task 3: Convert CSV to JSON

In [ ]:

df.to_json("students.json", orient="records", indent=4)
print("JSON file created")


JSON file created


## Task 4: Convert JSON to Avro

In [ ]:
# Import the writer function to serialize data into Avro format
# Import parse_schema to validate and optimize the Avro schema
from fastavro import writer, parse_schema

# Import the json module to read data from a JSON file
import json

# Define the Avro schema as a Python dictionary
# This schema describes the structure of each student record
# including field names and their corresponding data types
schema = {
    "type": "record",           # Indicates a complex Avro record type
    "name": "Student",          # Name of the record (schema name)
    "fields": [
        {"name": "student_id", "type": "int"},     # Integer student identifier
        {"name": "name", "type": "string"},        # Student name as a string
        {"name": "department", "type": "string"},  # Department name
        {"name": "marks", "type": "float"},        # Marks obtained (floating-point)
        {"name": "passed", "type": "boolean"}      # Pass/fail status
    ]
}

# Parse and validate the schema
# This step checks the schema for correctness and prepares it for efficient use
parsed_schema = parse_schema(schema)

# Open the JSON file containing student records in read mode
# The JSON file is expected to contain a list of dictionaries
# where each dictionary matches the structure defined in the Avro schema
with open("students.json") as f:
    records = json.load(f)

# Open an output file in binary write mode to store the Avro data
# Avro files are written in binary format for compact storage and fast access
with open("students.avro", "wb") as out:
    # Serialize the JSON records into Avro format using the parsed schema
    # Each record is written sequentially into the Avro file
    writer(out, parsed_schema, records)

# Print a confirmation message indicating successful Avro file creation
print("Avro file created")


Avro file created


## Note:
- Avro for compact storage, schema enforcement, and fast processing

- Commonly used in Big Data ecosystems (Kafka, Hadoop, Spark)

## Task 5: Convert Avro to Parquet

In [ ]:

# Import PyArrow, which provides in-memory columnar data structures
# and efficient interoperability with Parquet format
import pyarrow as pa

# Import the Parquet module from PyArrow for reading and writing Parquet files
import pyarrow.parquet as pq

# Import the Avro reader to deserialize Avro binary files
from fastavro import reader

# Open the Avro file in binary read mode
# Avro files store data in a compact binary format
with open("students.avro", "rb") as f:
    # Read all Avro records into a list of Python dictionaries
    # Each dictionary represents one row and follows the Avro schema
    avro_records = list(reader(f))

# Convert the list of dictionaries into a PyArrow Table
# PyArrow Tables store data in a columnar in-memory format,
# which is ideal for analytics and efficient compression
table = pa.Table.from_pylist(avro_records)

# Write the PyArrow Table to a Parquet file
# Parquet is a columnar, compressed, and analytics-optimized storage format
# Commonly used in data lakes and big data frameworks (Spark, Hive, Presto)
pq.write_table(table, "students.parquet")

# Print a confirmation message indicating successful Parquet file creation
print("Parquet file created")


Parquet file created


## Note:
- columnar, compressed, analytics-friendly
- A typical ETL pipeline step in big data systems
- Compression options

## Task 6: Data Validation

In [ ]:

df_parquet = pq.read_table("students.parquet").to_pandas()
df_parquet.head()


,student_id,name,department,marks,passed
0,1,Amit,CSE,85.500000,True
1,2,Riya,ECE,72.000000,True
2,3,Karan,ME,48.000000,False
3,4,Sneha,CSE,91.199997,True
